In [12]:
import pandas as pd
import numpy as np
from utils.funcs import get_variable_names, rename_columns, processed_diabetes_data, demographic_data2
from functools import reduce
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from imblearn.combine import SMOTEENN
import xgboost as xgb

In [13]:
def read_file(filename=''):

    dataset_path='raw_datasets'
    df=pd.read_sas(f'{dataset_path}/{filename}.XPT', format='xport')
    mapping=get_variable_names()
    df=rename_columns(df, mapping)  
    
    return df

In [14]:
df_diabetes=processed_diabetes_data()  
df_demographic=demographic_data2()
df_audio=read_file('audiometry')
df_blood_pressure=read_file('blood_pressure_cholesterol')
df_general_health=read_file('hospital_utilization_access_to_care')
df_weight=read_file('weight_history')
df_occupation=read_file('occupation')

In [15]:
dataframes = [df_diabetes,df_demographic, df_audio, df_blood_pressure, df_general_health, df_weight, df_occupation]
df = reduce(lambda left, right: pd.merge(left, right, on='sequence_no', how='inner'), dataframes)
columns_to_select = ['sequence_no', 'EverTold_Diabetes', 'gender', 'age','weight','weight2','HearingStatus_NoAid','EverTold_Hypertension','EverTold_HighCholesterol','GeneralHealth_Status','CurrentHeight','CurrentWeight','WorkExperience_LastWeek','WeightOneYearAgo']
df=df[columns_to_select]
df['bmi']=df['CurrentWeight']/(df['CurrentHeight']**2)
df = df.dropna()


In [16]:
y = df["EverTold_Diabetes"] 
X = df.drop(['EverTold_Diabetes','sequence_no'], axis=1)  
y=y.map({1.0: 1, 2.0: 0})

Reduced Features based on correlation analysis

In [17]:
X_reduced2=X.drop(['CurrentHeight', 'WeightOneYearAgo','CurrentWeight','gender'], axis=1)  


In [10]:
print(X_reduced2.columns)

Index(['age', 'weight', 'weight2', 'HearingStatus_NoAid',
       'EverTold_Hypertension', 'EverTold_HighCholesterol',
       'GeneralHealth_Status', 'WorkExperience_LastWeek', 'bmi'],
      dtype='object')


SVM on reduced features

In [11]:
def evaluate_with_kfold(X, y, sampler=None, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    svm = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)

    if sampler:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()),('sampler', sampler), ('svm', svm)])
    else:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()),('svm', svm)])
    
    auc_scores = []
    accuracy_scores = []

    for train_idx, valid_idx in skf.split(X, y):
        X_train, X_valid = X[train_idx], X[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        pipeline.fit(X_train, y_train)

        y_pred_proba = pipeline.predict_proba(X_valid)
        y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)

        auc_roc = roc_auc_score(y_valid, y_pred_proba[:, 1])
        accuracy = accuracy_score(y_valid, y_pred)

        auc_scores.append(auc_roc)
        accuracy_scores.append(accuracy)

    avg_auc = np.mean(auc_scores)
    avg_accuracy = np.mean(accuracy_scores)
    
    return avg_auc, avg_accuracy

X_np = np.array(X_reduced2) 
y_np = np.array(y)

auc_original, acc_original = evaluate_with_kfold(X_np, y_np)

smote = SMOTE(random_state=42)
auc_smote, acc_smote = evaluate_with_kfold(X_np, y_np, sampler=smote)

rus = RandomUnderSampler(random_state=42)
auc_rus, acc_rus = evaluate_with_kfold(X_np, y_np, sampler=rus)

smote_rus = SMOTETomek(random_state=42)
auc_smote_rus, acc_smote_rus = evaluate_with_kfold(X_np, y_np, sampler=smote_rus)

print("Summary of Results")
print(f"Original Data AUC-ROC: {auc_original:.4f}, Accuracy: {acc_original:.4f}")
print(f"SMOTE AUC-ROC: {auc_smote:.4f}, Accuracy: {acc_smote:.4f}")
print(f"RUS AUC-ROC: {auc_rus:.4f}, Accuracy: {acc_rus:.4f}")
print(f"SMOTE + RUS AUC-ROC: {auc_smote_rus:.4f}, Accuracy: {acc_smote_rus:.4f}")



### Summary of Results ###
Original Data AUC-ROC: 0.8230, Accuracy: 0.8687
SMOTE AUC-ROC: 0.8172, Accuracy: 0.7380
RUS AUC-ROC: 0.8260, Accuracy: 0.7151
SMOTE + RUS AUC-ROC: 0.8175, Accuracy: 0.7370


### Random Forest Classifier 

In [12]:
def evaluate_with_kfold_rf(X, y, sampler=None, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    
    if sampler:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('sampler', sampler), ('rf', rf)])
    else:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('rf', rf)])
    
    auc_scores = []
    accuracy_scores = []

    for train_idx, valid_idx in skf.split(X, y):
        X_train, X_valid = X[train_idx], X[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        pipeline.fit(X_train, y_train)

        y_pred_proba = pipeline.predict_proba(X_valid)
        y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)

        auc_roc = roc_auc_score(y_valid, y_pred_proba[:, 1])
        accuracy = accuracy_score(y_valid, y_pred)

        auc_scores.append(auc_roc)
        accuracy_scores.append(accuracy)

    avg_auc = np.mean(auc_scores)
    avg_accuracy = np.mean(accuracy_scores)
    
    return avg_auc, avg_accuracy

X_np = np.array(X)
y_np = np.array(y)

auc_original, acc_original = evaluate_with_kfold_rf(X_np, y_np)

smote = SMOTE(random_state=42)
auc_smote, acc_smote = evaluate_with_kfold_rf(X_np, y_np, sampler=smote)

rus = RandomUnderSampler(random_state=42)
auc_rus, acc_rus = evaluate_with_kfold_rf(X_np, y_np, sampler=rus)

smote_rus = SMOTETomek(random_state=42)
auc_smote_rus, acc_smote_rus = evaluate_with_kfold_rf(X_np, y_np, sampler=smote_rus)

print("Summary of Results")
print(f"Original Data AUC-ROC: {auc_original:.4f}, Accuracy: {acc_original:.4f}")
print(f"SMOTE AUC-ROC: {auc_smote:.4f}, Accuracy: {acc_smote:.4f}")
print(f"RUS AUC-ROC: {auc_rus:.4f}, Accuracy: {acc_rus:.4f}")
print(f"SMOTE + RUS AUC-ROC: {auc_smote_rus:.4f}, Accuracy: {acc_smote_rus:.4f}")



### Summary of Results ###
Original Data AUC-ROC: 0.8322, Accuracy: 0.8726
SMOTE AUC-ROC: 0.8311, Accuracy: 0.8340
RUS AUC-ROC: 0.8323, Accuracy: 0.7309
SMOTE + RUS AUC-ROC: 0.8326, Accuracy: 0.8318


Reduced features using correlation matrix

In [13]:
def evaluate_with_kfold_rf(X, y, sampler=None, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    
    if sampler:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('sampler', sampler), ('rf', rf)])
    else:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('rf', rf)])
    
    auc_scores = []
    accuracy_scores = []

    for train_idx, valid_idx in skf.split(X, y):
        X_train, X_valid = X[train_idx], X[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        pipeline.fit(X_train, y_train)

        y_pred_proba = pipeline.predict_proba(X_valid)
        y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)

        auc_roc = roc_auc_score(y_valid, y_pred_proba[:, 1])
        accuracy = accuracy_score(y_valid, y_pred)

        auc_scores.append(auc_roc)
        accuracy_scores.append(accuracy)

    avg_auc = np.mean(auc_scores)
    avg_accuracy = np.mean(accuracy_scores)
    
    return avg_auc, avg_accuracy

X_np = np.array(X_reduced2)  
y_np = np.array(y)

auc_original, acc_original = evaluate_with_kfold_rf(X_np, y_np)

smote = SMOTE(random_state=42)
auc_smote, acc_smote = evaluate_with_kfold_rf(X_np, y_np, sampler=smote)

rus = RandomUnderSampler(random_state=42)
auc_rus, acc_rus = evaluate_with_kfold_rf(X_np, y_np, sampler=rus)

smote_rus = SMOTETomek(random_state=42)
auc_smote_rus, acc_smote_rus = evaluate_with_kfold_rf(X_np, y_np, sampler=smote_rus)

print("Summary of Results")
print(f"Original Data AUC-ROC: {auc_original:.4f}, Accuracy: {acc_original:.4f}")
print(f"SMOTE AUC-ROC: {auc_smote:.4f}, Accuracy: {acc_smote:.4f}")
print(f"RUS AUC-ROC: {auc_rus:.4f}, Accuracy: {acc_rus:.4f}")
print(f"SMOTE + RUS AUC-ROC: {auc_smote_rus:.4f}, Accuracy: {acc_smote_rus:.4f}")



### Summary of Results ###
Original Data AUC-ROC: 0.8312, Accuracy: 0.8681
SMOTE AUC-ROC: 0.8239, Accuracy: 0.8272
RUS AUC-ROC: 0.8283, Accuracy: 0.7345
SMOTE + RUS AUC-ROC: 0.8260, Accuracy: 0.8266


Reduced features using Recursive Feature Elimination

In [19]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rfe = RFE(estimator=rf, n_features_to_select=9) 
X_reduced = rfe.fit_transform(X, y)

print("Selected Features:", X.columns[rfe.support_])

Selected Features: Index(['age', 'weight', 'weight2', 'EverTold_Hypertension',
       'GeneralHealth_Status', 'CurrentHeight', 'CurrentWeight',
       'WeightOneYearAgo', 'bmi'],
      dtype='object')


In [20]:
def evaluate_with_kfold_rf(X, y, sampler=None, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    
    if sampler:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('sampler', sampler), ('rf', rf)])
    else:
        pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('rf', rf)])
    
    auc_scores = []
    accuracy_scores = []

    for train_idx, valid_idx in skf.split(X, y):
        X_train, X_valid = X[train_idx], X[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        pipeline.fit(X_train, y_train)

        y_pred_proba = pipeline.predict_proba(X_valid)
        y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)

        auc_roc = roc_auc_score(y_valid, y_pred_proba[:, 1])
        accuracy = accuracy_score(y_valid, y_pred)

        auc_scores.append(auc_roc)
        accuracy_scores.append(accuracy)

    avg_auc = np.mean(auc_scores)
    avg_accuracy = np.mean(accuracy_scores)
    
    return avg_auc, avg_accuracy

X_np = np.array(X_reduced)  
y_np = np.array(y)

auc_original, acc_original = evaluate_with_kfold_rf(X_np, y_np)

smote = SMOTE(random_state=42)
auc_smote, acc_smote = evaluate_with_kfold_rf(X_np, y_np, sampler=smote)

rus = RandomUnderSampler(random_state=42)
auc_rus, acc_rus = evaluate_with_kfold_rf(X_np, y_np, sampler=rus)

smote_rus = SMOTETomek(random_state=42)
auc_smote_rus, acc_smote_rus = evaluate_with_kfold_rf(X_np, y_np, sampler=smote_rus)

print("Summary of Results")
print(f"Original Data AUC-ROC: {auc_original:.4f}, Accuracy: {acc_original:.4f}")
print(f"SMOTE AUC-ROC: {auc_smote:.4f}, Accuracy: {acc_smote:.4f}")
print(f"RUS AUC-ROC: {auc_rus:.4f}, Accuracy: {acc_rus:.4f}")
print(f"SMOTE + RUS AUC-ROC: {auc_smote_rus:.4f}, Accuracy: {acc_smote_rus:.4f}")



### Summary of Results ###
Original Data AUC-ROC: 0.8144, Accuracy: 0.8676
SMOTE AUC-ROC: 0.8107, Accuracy: 0.8079
RUS AUC-ROC: 0.8117, Accuracy: 0.7163
SMOTE + RUS AUC-ROC: 0.8126, Accuracy: 0.8056


### XGBoost

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_np = np.array(X_scaled)  
y_np = np.array(y)

sampling_methods = {
    "no_sampling": None,
    "smote": SMOTE(random_state=42),
    "rus": RandomUnderSampler(random_state=42),
    "smote_rus": SMOTEENN(random_state=42)
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {method: [] for method in sampling_methods.keys()}

for method, sampler in sampling_methods.items():
    for train_idx, val_idx in kfold.split(X_np, y_np):
        X_train, X_val = X_np[train_idx], X_np[val_idx]
        y_train, y_val = y_np[train_idx], y_np[val_idx]

        if sampler is not None:
            X_train, y_train = sampler.fit_resample(X_train, y_train)

        model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        y_pred_proba = model.predict_proba(X_val)[:, 1]

        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)
        roc_auc = roc_auc_score(y_val, y_pred_proba)

        results[method].append({"accuracy": acc, "f1": f1, "roc_auc": roc_auc})

for method, scores in results.items():
    avg_acc = np.mean([s["accuracy"] for s in scores])
    avg_f1 = np.mean([s["f1"] for s in scores])
    avg_roc_auc = np.mean([s["roc_auc"] for s in scores])
    print(f"\nMethod: {method}")
    print(f"Average Accuracy: {avg_acc:.4f}")
    print(f"Average F1 Score: {avg_f1:.4f}")
    print(f"Average ROC AUC: {avg_roc_auc:.4f}")



Method: no_sampling
Average Accuracy: 0.8687
Average F1 Score: 0.3365
Average ROC AUC: 0.8229

Method: smote
Average Accuracy: 0.8551
Average F1 Score: 0.3864
Average ROC AUC: 0.8206

Method: rus
Average Accuracy: 0.7292
Average F1 Score: 0.4214
Average ROC AUC: 0.8127

Method: smote_rus
Average Accuracy: 0.8050
Average F1 Score: 0.4672
Average ROC AUC: 0.8319


Reduced features

In [24]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reduced2)
X_np = np.array(X_scaled)  
y_np = np.array(y)

sampling_methods = {
    "no_sampling": None,
    "smote": SMOTE(random_state=42),
    "rus": RandomUnderSampler(random_state=42),
    "smote_rus": SMOTEENN(random_state=42)
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {method: [] for method in sampling_methods.keys()}

for method, sampler in sampling_methods.items():
    for train_idx, val_idx in kfold.split(X_np, y_np):
        X_train, X_val = X_np[train_idx], X_np[val_idx]
        y_train, y_val = y_np[train_idx], y_np[val_idx]

        if sampler is not None:
            X_train, y_train = sampler.fit_resample(X_train, y_train)

        model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        y_pred_proba = model.predict_proba(X_val)[:, 1]

        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)
        roc_auc = roc_auc_score(y_val, y_pred_proba)

        results[method].append({"accuracy": acc, "f1": f1, "roc_auc": roc_auc})

for method, scores in results.items():
    avg_acc = np.mean([s["accuracy"] for s in scores])
    avg_f1 = np.mean([s["f1"] for s in scores])
    avg_roc_auc = np.mean([s["roc_auc"] for s in scores])
    print(f"\nMethod: {method}")
    print(f"Average Accuracy: {avg_acc:.4f}")
    print(f"Average F1 Score: {avg_f1:.4f}")
    print(f"Average ROC AUC: {avg_roc_auc:.4f}")



Method: no_sampling
Average Accuracy: 0.8648
Average F1 Score: 0.3100
Average ROC AUC: 0.8174

Method: smote
Average Accuracy: 0.8336
Average F1 Score: 0.3970
Average ROC AUC: 0.8141

Method: rus
Average Accuracy: 0.7261
Average F1 Score: 0.4168
Average ROC AUC: 0.8102

Method: smote_rus
Average Accuracy: 0.7921
Average F1 Score: 0.4578
Average ROC AUC: 0.8294
